In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Imports, config, W&B

In [2]:
import re, numpy as np, pandas as pd, torch
import torch.nn as nn
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

DATA    = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OPTIONS = ["A", "B", "C", "D", "E"]
device  = "cuda" if torch.cuda.is_available() else "cpu"

EMBED_DIM, HIDDEN = 128, 128
MAX_P, MAX_O      = 64, 64          
BATCH, EPOCHS, LR = 32, 10, 1e-3
MIN_FREQ          = 2
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

train = pd.read_csv(f"{DATA}/train.csv")
test  = pd.read_csv(f"{DATA}/test.csv")
train["label"] = train["answer"].map({c: i for i, c in enumerate(OPTIONS)})
print("device:", device, "| train:", train.shape)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ishankgpt02 (ishankgpt02-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


device: cuda | train: (2000, 9)


# From-scratch tokenizer + vocab

In [3]:
def tokenize(t):
    return re.findall(r"[a-z0-9]+", str(t).lower())

counter = Counter()
for col in ["prompt"] + OPTIONS:
    for t in train[col]:
        counter.update(tokenize(t))

PAD, UNK = 0, 1
vocab = {"<pad>": PAD, "<unk>": UNK}
for w, c in counter.items():
    if c >= MIN_FREQ:
        vocab[w] = len(vocab)
VOCAB_SIZE = len(vocab)
print("vocab size:", VOCAB_SIZE)

def encode(text, maxlen):
    ids = [vocab.get(w, UNK) for w in tokenize(text)][:maxlen]
    return ids + [PAD] * (maxlen - len(ids))

vocab size: 2972


# Dataset + loaders

In [4]:
class MCQDataset(Dataset):
    def __init__(self, df, has_label=True):
        self.p = [encode(t, MAX_P) for t in df["prompt"]]
        self.o = [[encode(str(r[c]), MAX_O) for c in OPTIONS] for _, r in df.iterrows()]
        self.y = df["label"].tolist() if has_label else [0]*len(df)
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        return (torch.tensor(self.p[i]), torch.tensor(self.o[i]), torch.tensor(self.y[i]))

tr_df, va_df = train_test_split(train, test_size=0.1, random_state=SEED, stratify=train["label"])
tr_loader = DataLoader(MCQDataset(tr_df), batch_size=BATCH, shuffle=True)
va_loader = DataLoader(MCQDataset(va_df), batch_size=64)
te_loader = DataLoader(MCQDataset(test, has_label=False), batch_size=64)
print("batches/epoch:", len(tr_loader))

batches/epoch: 57


# BiLSTM model

In [5]:
class BiLSTMMatcher(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)   
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True, bidirectional=True)
        self.scorer = nn.Sequential(
            nn.Linear(hidden*2*4, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, 1))

    def encode(self, ids):                      
        emb = self.embedding(ids)               
        out, _ = self.lstm(emb)                 
        mask = (ids != PAD).unsqueeze(-1).float()
        pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1)   
        return pooled

    def forward(self, prompt, options):         
        p = self.encode(prompt)
        B, N, To = options.shape
        o = self.encode(options.view(B*N, To)).view(B, N, -1)
        p = p.unsqueeze(1).expand(-1, N, -1)
        feat = torch.cat([p, o, p*o, (p-o).abs()], dim=-1)        
        return self.scorer(feat).squeeze(-1)                      

model = BiLSTMMatcher(VOCAB_SIZE, EMBED_DIM, HIDDEN).to(device)
print("params:", sum(p.numel() for p in model.parameters()))

params: 775937


# W&B run + metric axes

In [6]:
run = wandb.init(entity="ishankgpt02-na", project="23f1002033-t22026",
                 name="bilstm-from-scratch",
                 config={"embed_dim": EMBED_DIM, "hidden": HIDDEN, "epochs": EPOCHS,
                         "lr": LR, "batch": BATCH, "vocab": VOCAB_SIZE, "scratch": True})

wandb.define_metric("global_step")
wandb.define_metric("train/*", step_metric="global_step")
wandb.define_metric("epoch")
wandb.define_metric("val/*", step_metric="epoch")

def map3(scores, labels):
    order = np.argsort(-scores, axis=1)
    s = 0.0
    for o, l in zip(order, labels):
        pos = int(np.where(o == l)[0][0])
        if pos < 3: s += 1.0/(pos+1)
    return s/len(labels)

# Training loop

In [7]:
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS*len(tr_loader))
lossfn = nn.CrossEntropyLoss()
gstep = 0

for epoch in range(1, EPOCHS+1):
    model.train(); ep_loss = 0
    for prompt, options, y in tr_loader:
        prompt, options, y = prompt.to(device), options.to(device), y.to(device)
        opt.zero_grad()
        logits = model(prompt, options)
        loss = lossfn(logits, y)
        loss.backward(); opt.step(); sched.step()
        gstep += 1; ep_loss += loss.item()
        # STEP-LEVEL log (train loss curve)
        wandb.log({"train/loss": loss.item(),
                   "train/lr": sched.get_last_lr()[0],
                   "global_step": gstep})

    model.eval(); all_scores, all_y, vloss = [], [], 0
    with torch.no_grad():
        for prompt, options, y in va_loader:
            logits = model(prompt.to(device), options.to(device))
            vloss += lossfn(logits, y.to(device)).item()
            all_scores.append(logits.cpu().numpy()); all_y.append(y.numpy())
    scores = np.concatenate(all_scores); ys = np.concatenate(all_y)
    val_map3 = map3(scores, ys)
    val_acc  = float((scores.argmax(1) == ys).mean())
    wandb.log({"val/loss": vloss/len(va_loader), "val/map@3": val_map3,
               "val/acc": val_acc, "train/epoch_loss": ep_loss/len(tr_loader),
               "epoch": epoch})
    print(f"Epoch {epoch:2d} | train_loss {ep_loss/len(tr_loader):.4f} "
          f"| val_loss {vloss/len(va_loader):.4f} | val MAP@3 {val_map3:.4f} | acc {val_acc:.4f}")

Epoch  1 | train_loss 1.2867 | val_loss 0.6229 | val MAP@3 0.9183 | acc 0.8600
Epoch  2 | train_loss 0.2787 | val_loss 0.0471 | val MAP@3 0.9975 | acc 0.9950
Epoch  3 | train_loss 0.0324 | val_loss 0.0044 | val MAP@3 1.0000 | acc 1.0000
Epoch  4 | train_loss 0.0103 | val_loss 0.0032 | val MAP@3 1.0000 | acc 1.0000
Epoch  5 | train_loss 0.0030 | val_loss 0.0021 | val MAP@3 1.0000 | acc 1.0000
Epoch  6 | train_loss 0.0022 | val_loss 0.0009 | val MAP@3 1.0000 | acc 1.0000
Epoch  7 | train_loss 0.0007 | val_loss 0.0007 | val MAP@3 1.0000 | acc 1.0000
Epoch  8 | train_loss 0.0006 | val_loss 0.0007 | val MAP@3 1.0000 | acc 1.0000
Epoch  9 | train_loss 0.0005 | val_loss 0.0007 | val MAP@3 1.0000 | acc 1.0000
Epoch 10 | train_loss 0.0007 | val_loss 0.0006 | val MAP@3 1.0000 | acc 1.0000


# Test submission

In [8]:
model.eval(); test_scores = []
with torch.no_grad():
    for prompt, options, _ in te_loader:
        logits = model(prompt.to(device), options.to(device))
        test_scores.append(logits.cpu().numpy())
test_scores = np.concatenate(test_scores)
order = np.argsort(-test_scores, axis=1)
preds = [" ".join(OPTIONS[j] for j in row[:3]) for row in order]
pd.DataFrame({"ID": test["id"], "Prediction": preds}).to_csv("submission.csv", index=False)
run.summary["final_val_map@3"] = val_map3
run.finish()
print("submission.csv saved | final val MAP@3:", round(val_map3, 4))

epoch,▁▂▃▃▄▅▆▆▇█
global_step,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇█████████
train/epoch_loss,█▃▁▁▁▁▁▁▁▁
train/loss,████▄▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,█████▇▇▇▇▇▇▇▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
val/acc,▁█████████
val/loss,█▂▁▁▁▁▁▁▁▁
val/map@3,▁█████████
epoch,10
final_val_map@3,1
global_step,570


submission.csv saved | final val MAP@3: 1.0
